# **CSC491 Final Project**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_absolute_error

### Import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/mmfit_setwise_left_watch_50hz.npz"
data = np.load(path, allow_pickle=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X = data["X"]                  # (616, 1838, 6)
y = data["y"]                  # (616,)
reps = data["reps"].astype(np.float32)
split = data["split_repo_unseen"]
label_names = data["label_names"]

print("Raw X shape:", X.shape)

# Flatten sequence
X_flat = X.reshape(X.shape[0], -1).astype(np.float32)
print("Flattened X shape:", X_flat.shape)   # (616, 11028)

# Split
train_mask = split == "train"
val_mask = split == "val"
test_mask = split == "test"

X_train, X_val, X_test = X_flat[train_mask], X_flat[val_mask], X_flat[test_mask]
y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]
r_train, r_val, r_test = reps[train_mask], reps[val_mask], reps[test_mask]

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print(X_train.shape, X_val.shape, X_test.shape)


Raw X shape: (616, 1838, 6)
Flattened X shape: (616, 11028)
(301, 11028) (86, 11028) (139, 11028)


In [ ]:
class ExerciseDataset(Dataset):
    def __init__(self, X, y, reps):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.reps = torch.tensor(reps, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.reps[idx]

train_ds = ExerciseDataset(X_train, y_train, r_train)
val_ds = ExerciseDataset(X_val, y_val, r_val)
test_ds = ExerciseDataset(X_test, y_test, r_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)


class FlattenedMultiTaskMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.class_head = nn.Linear(128, num_classes)
        self.rep_head = nn.Linear(128, 1)

    def forward(self, x):
        h = self.backbone(x)
        class_logits = self.class_head(h)
        rep_pred = self.rep_head(h)
        return class_logits, rep_pred

device = "cuda" if torch.cuda.is_available() else "cpu"
model = FlattenedMultiTaskMLP(
    input_dim=X_train.shape[1],
    num_classes=len(label_names)
).to(device)

cls_loss_fn = nn.CrossEntropyLoss()
rep_loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

Total Parameters: 5,812,491


In [ ]:
def run_epoch(loader, train=False):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_y, all_yhat = [], []
    all_r, all_rhat = [], []

    for xb, yb, rb in loader:
        xb, yb, rb = xb.to(device), yb.to(device), rb.to(device)

        with torch.set_grad_enabled(train):
            class_logits, rep_pred = model(xb)

            cls_loss = cls_loss_fn(class_logits, yb)
            rep_loss = rep_loss_fn(rep_pred, rb)
            loss = cls_loss + 0.2 * rep_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * xb.size(0)

        all_y.append(yb.detach().cpu().numpy())
        all_yhat.append(torch.argmax(class_logits, dim=1).detach().cpu().numpy())
        all_r.append(rb.detach().cpu().numpy().ravel())
        all_rhat.append(rep_pred.detach().cpu().numpy().ravel())

    all_y = np.concatenate(all_y)
    all_yhat = np.concatenate(all_yhat)
    all_r = np.concatenate(all_r)
    all_rhat = np.concatenate(all_rhat)

    avg_loss = total_loss / len(loader.dataset)
    acc = (all_y == all_yhat).mean()
    mae = mean_absolute_error(all_r, all_rhat)

    return avg_loss, acc, mae, all_y, all_yhat, all_r, all_rhat

In [ ]:
best_val_loss = float("inf")
best_state = None

for epoch in range(100):
    train_loss, train_acc, train_mae, *_ = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_mae, *_ = run_epoch(val_loader, train=False)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1:02d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} mae {train_mae:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} mae {val_mae:.4f}"
        )

model.load_state_dict(best_state)


Epoch 01 | train loss 2.9988 acc 0.4319 mae 6.7429 | val loss 1.7205 acc 0.7907 mae 4.0762
Epoch 10 | train loss 0.4578 acc 0.9867 mae 2.0020 | val loss 0.9152 acc 0.8488 mae 1.5617
Epoch 20 | train loss 0.3957 acc 0.9867 mae 1.6554 | val loss 0.8548 acc 0.8605 mae 1.3808
Epoch 30 | train loss 0.3476 acc 0.9900 mae 1.6695 | val loss 1.2887 acc 0.8488 mae 1.4372
Epoch 40 | train loss 0.3364 acc 0.9967 mae 1.6377 | val loss 1.2932 acc 0.9070 mae 1.8968
Epoch 50 | train loss 0.3920 acc 0.9900 mae 1.8041 | val loss 1.6334 acc 0.8721 mae 1.6603
Epoch 60 | train loss 0.3502 acc 0.9900 mae 1.6343 | val loss 1.9405 acc 0.8721 mae 1.7847
Epoch 70 | train loss 0.4941 acc 0.9701 mae 1.6352 | val loss 2.8264 acc 0.8953 mae 1.3816
Epoch 80 | train loss 0.3479 acc 1.0000 mae 1.7232 | val loss 2.6028 acc 0.8837 mae 1.4892
Epoch 90 | train loss 0.3806 acc 0.9934 mae 1.7169 | val loss 2.2292 acc 0.9186 mae 1.8582
Epoch 100 | train loss 0.3416 acc 0.9967 mae 1.6319 | val loss 2.8751 acc 0.8953 mae 1.633

<All keys matched successfully>

In [ ]:
test_loss, test_acc, test_mae, y_true, y_pred, r_true, r_pred = run_epoch(test_loader, train=False)

print("Test accuracy:", test_acc)
print("Test rep MAE:", test_mae)
print("Off-by-one rep accuracy:", np.mean(np.abs(np.round(r_pred) - r_true) <= 1))

print("\nClassification report")
print(classification_report(y_true, y_pred, target_names=label_names))


Test accuracy: 0.7553956834532374
Test rep MAE: 2.2669272422790527
Off-by-one rep accuracy: 0.381294964028777

Classification report
                         precision    recall  f1-score   support

                 squats       0.82      0.60      0.69        15
                 lunges       0.87      0.87      0.87        15
            bicep_curls       0.67      0.15      0.25        13
                 situps       0.77      0.67      0.71        15
                pushups       0.78      0.93      0.85        15
      tricep_extensions       0.71      0.80      0.75        15
          dumbbell_rows       0.58      0.93      0.72        15
          jumping_jacks       0.91      0.83      0.87        12
dumbbell_shoulder_press       1.00      1.00      1.00        12
lateral_shoulder_raises       0.60      0.75      0.67        12

               accuracy                           0.76       139
              macro avg       0.77      0.75      0.74       139
           weighted 

In [ ]:

# Pick one test sample with reps != 10
candidate_idxs = np.where((test_mask) & (reps != 10))[0]
print("num test samples with reps != 10:", len(candidate_idxs))

idx = candidate_idxs[0]   # change this if you want a different sample

# Prepare one sample for the flattened MLP
x_single = X_flat[idx:idx+1]                 # shape (1, 11028)
x_single = scaler.transform(x_single).astype(np.float32)
x_single_t = torch.tensor(x_single, dtype=torch.float32).to(device)

# Inference
model.eval()
with torch.no_grad():
    class_logits, rep_pred = model(x_single_t)
    pred_class = torch.argmax(class_logits, dim=1).item()
    pred_reps = rep_pred.item()

print("Sample index:", idx)
print("True label:", label_names[y[idx]])
print("Predicted label:", label_names[pred_class])
print("True reps:", reps[idx])
print("Predicted reps (raw):", pred_reps)
print("Predicted reps (rounded):", int(round(pred_reps)))

num test samples with reps != 10: 20
Sample index: 3
True label: pushups
Predicted label: pushups
True reps: 11.0
Predicted reps (raw): 9.195571899414062
Predicted reps (rounded): 9


In [ ]:
model.eval()
shown = 0

for idx in np.where((test_mask) & (reps != 10))[0]:
    x_single = X_flat[idx:idx+1]
    x_single = scaler.transform(x_single).astype(np.float32)
    x_single_t = torch.tensor(x_single, dtype=torch.float32).to(device)

    with torch.no_grad():
        class_logits, rep_pred = model(x_single_t)
        pred_class = torch.argmax(class_logits, dim=1).item()
        pred_reps = rep_pred.item()

    print(f"idx={idx} | true_ex={label_names[y[idx]]} | pred_ex={label_names[pred_class]} | "
          f"true_reps={reps[idx]} | pred_reps={pred_reps:.2f} | rounded={int(round(pred_reps))}")

    shown += 1
    if shown == 10:
        break
